# TM-02 RSBP: Smart Laboratory Access Control

Notebook ini menerjemahkan kebijakan pada slide 29-35 menjadi propositional logic dan memeriksa lima skenario menggunakan Z3.

## Setup

Sel ini mengimpor Z3. Jalankan terlebih dahulu sebelum menjalankan skenario.

In [3]:
from z3 import *

## Proposisi dan Knowledge Base

Kartu aktif, pelatihan selesai, dan tidak sedang terkena sanksi adalah syarat dasar. Akses di luar jam kerja membutuhkan izin khusus dan supervisor. Penggunaan alat membutuhkan akses lab, sudah berada di dalam lab, dan sudah menyelesaikan pelatihan. Alat berisiko tinggi juga membutuhkan sertifikasi dan supervisor.

In [4]:
K, T, S, W, P, V, C, D, E, O, N, A, I, U, H = Bools(
    "K T S W P V C D E O N A I U H"
)

# K: kartu aktif, T: pelatihan selesai, S: terkena sanksi
# W: jam kerja normal, P: izin khusus, V: supervisor hadir
# C: sertifikasi alat berisiko, D: kondisi darurat
# E: kartu kedaluwarsa, O: petugas keselamatan
# N: akses lab, A: emergency access, I: berada di dalam lab
# U: boleh menggunakan alat, H: boleh menggunakan alat berisiko tinggi
aturan = [
    N == Or(
        And(K, T, Not(S), Not(E), Not(D), Or(W, And(P, V))),
        A,
    ),
    Implies(E, Not(K)),
    A == And(O, D),
    U == And(N, I, T),
    H == And(U, C, V),
]

def buat_kb(*fakta):
    solver = Solver()
    solver.add(*aturan)
    solver.add(*fakta)
    return solver

def cek_query(solver, query):
    solver.push()
    solver.add(query)
    hasil = solver.check()
    solver.pop()
    return hasil

def tampilkan_status(solver, query, nama):
    hasil_negasi = cek_query(solver, Not(query))
    hasil_positif = cek_query(solver, query)
    print(f"  KB AND Not({query}): {hasil_negasi}")
    print(f"  KB AND {query}: {hasil_positif}")
    if hasil_negasi == unsat:
        print(f"  => {nama}: TERBUKTI BENAR")
    elif hasil_positif == unsat:
        print(f"  => {nama}: TERBUKTI SALAH")
    else:
        print(f"  => {nama}: BELUM DAPAT DIPASTIKAN")

## Skenario 1: Akses Normal

Mahasiswa memiliki kartu aktif, sudah mengikuti pelatihan, tidak terkena sanksi, mengakses pada jam kerja normal, dan laboratorium tidak darurat.

In [5]:
s1 = buat_kb(K, T, Not(S), W, Not(D), Not(E))
print("Konsistensi KB:", s1.check())
tampilkan_status(s1, N, "Akses laboratorium")

Konsistensi KB: sat
  KB AND Not(N): unsat
  KB AND N: sat
  => Akses laboratorium: TERBUKTI BENAR


## Skenario 2: Akses di Luar Jam Kerja

Mahasiswa memiliki izin khusus, tetapi fakta kehadiran supervisor belum diberikan. Kita mencari counterexample untuk melihat apakah akses dapat dipastikan.

In [6]:
s2 = buat_kb(K, T, Not(S), Not(W), P, Not(D), Not(E))
print("Konsistensi KB:", s2.check())
tampilkan_status(s2, N, "Akses laboratorium")
s2.push()
s2.add(Not(N))
s2.check()
print("Counterexample:", s2.model())
s2.pop()

s2.push()
s2.add(V)
tampilkan_status(s2, N, "Akses laboratorium (+ supervisor hadir)")
s2.pop()

Konsistensi KB: sat
  KB AND Not(N): sat
  KB AND N: sat
  => Akses laboratorium: BELUM DAPAT DIPASTIKAN
Counterexample: [H = False,
 V = False,
 K = True,
 S = False,
 W = False,
 U = False,
 P = True,
 T = True,
 D = False,
 E = False,
 N = False,
 A = False]
  KB AND Not(N): unsat
  KB AND N: sat
  => Akses laboratorium (+ supervisor hadir): TERBUKTI BENAR


## Skenario 3: Akses Lab dan Alat Berisiko Tinggi

Mahasiswa boleh masuk dan sudah berada di dalam lab, tetapi tidak memiliki sertifikasi alat berisiko tinggi.

In [7]:
s3 = buat_kb(K, T, Not(S), W, Not(C), Not(D), Not(E), I)
print("Konsistensi KB:", s3.check())
tampilkan_status(s3, N, "Akses laboratorium")
tampilkan_status(s3, H, "Penggunaan alat berisiko tinggi")
tampilkan_status(s3, And(N, Not(H)), "Boleh masuk tapi tanpa alat berisiko")

Konsistensi KB: sat
  KB AND Not(N): unsat
  KB AND N: sat
  => Akses laboratorium: TERBUKTI BENAR
  KB AND Not(H): sat
  KB AND H: unsat
  => Penggunaan alat berisiko tinggi: TERBUKTI SALAH
  KB AND Not(And(N, Not(H))): unsat
  KB AND And(N, Not(H)): sat
  => Boleh masuk tapi tanpa alat berisiko: TERBUKTI BENAR


## Skenario 4: Penggunaan Alat Biasa

Mahasiswa berada di dalam lab dan supervisor hadir. Sertifikasi alat berisiko tinggi tidak dimiliki, tetapi skenario ini hanya menanyakan penggunaan alat biasa.

In [8]:
s4 = buat_kb(K, T, Not(S), W, Not(C), V, Not(D), Not(E), I)
print("Konsistensi KB:", s4.check())
tampilkan_status(s4, U, "Penggunaan alat biasa")
tampilkan_status(s4, H, "Penggunaan alat berisiko tinggi")
print("Model:", s4.model())

Konsistensi KB: sat
  KB AND Not(U): unsat
  KB AND U: sat
  => Penggunaan alat biasa: TERBUKTI BENAR
  KB AND Not(H): sat
  KB AND H: unsat
  => Penggunaan alat berisiko tinggi: TERBUKTI SALAH
Model: [I = True,
 H = False,
 K = True,
 S = False,
 W = True,
 V = True,
 U = True,
 E = False,
 T = True,
 D = False,
 A = False,
 N = True,
 C = False]


## Skenario 5: Inkonsistensi Database

Database mahasiswa menyatakan pelatihan selesai, sedangkan safety office menyatakan pelatihan belum selesai. Unsat core digunakan untuk menunjukkan sumber konflik.

In [9]:
s5 = Solver()
s5.set(unsat_core=True)
s5.add(aturan)
s5.assert_and_track(T, "database_mahasiswa")
s5.assert_and_track(Not(T), "safety_office")

print("Konsistensi KB:", s5.check())
print("Penyebab konflik:", s5.unsat_core())

kb_tanpa_konflik = Solver()
kb_tanpa_konflik.add(aturan)
print("KB tanpa fakta konflik:", kb_tanpa_konflik.check())

Konsistensi KB: unsat
Penyebab konflik: [database_mahasiswa, safety_office]
KB tanpa fakta konflik: sat
